In [1]:
# Cellule 1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import xgboost as xgb
import joblib
import warnings
warnings.filterwarnings("ignore")

dataset = pd.read_csv("../data/processed/dataset_communal.csv")

# Exclure Cotonou (Littoral) comme on avait fait avec le dataset départemental
dataset = dataset[dataset["departement"] != "Littoral"].copy().reset_index(drop=True)

print(f"Shape : {dataset.shape}")
print(f"Communes  : {dataset['commune'].nunique()}")
print(f"Valeurs manquantes : {dataset.isnull().sum().sum()}")

Shape : (1514, 117)
Communes  : 76
Valeurs manquantes : 0


In [2]:
# Cellule 2 — préparation features et cible
cols_exclure = ["commune", "departement", "type_sol", "rendement_kg_ha", "rendement_t_ha"]
X = dataset.drop(columns=cols_exclure)
y = dataset["rendement_t_ha"]

print(f"Features X : {X.shape}")
print(f"Cible y    : {y.shape}")
print(f"Extrait colonnes : {X.columns.tolist()[:10]}")

Features X : (1514, 112)
Cible y    : (1514,)
Extrait colonnes : ['annee', 'precip_01_mm', 'temp_moy_01_c', 'temp_max_01_c', 'temp_min_01_c', 'humidite_01_pct', 'rayonnement_01', 'precip_02_mm', 'temp_moy_02_c', 'temp_max_02_c']


In [3]:
# Cellule 3 — séparation train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train : {X_train.shape} — {len(y_train)} observations")
print(f"Test  : {X_test.shape} — {len(y_test)} observations")
print(f"\nDistribution du rendement :")
print(f"Train — moyenne : {y_train.mean():.3f} t/ha, std : {y_train.std():.3f}")
print(f"Test  — moyenne : {y_test.mean():.3f} t/ha, std : {y_test.std():.3f}")

Train : (1211, 112) — 1211 observations
Test  : (303, 112) — 303 observations

Distribution du rendement :
Train — moyenne : 1.213 t/ha, std : 0.410
Test  — moyenne : 1.167 t/ha, std : 0.404


In [4]:
# Cellule 4 — fonction d'évaluation
def evaluer_modele(nom, modele, X_train, X_test, y_train, y_test):
    modele.fit(X_train, y_train)
    y_pred_train = modele.predict(X_train)
    y_pred_test  = modele.predict(X_test)

    resultats = {
        "Modele":     nom,
        "RMSE_train": np.sqrt(mean_squared_error(y_train, y_pred_train)),
        "RMSE_test":  np.sqrt(mean_squared_error(y_test,  y_pred_test)),
        "R2_train":   r2_score(y_train, y_pred_train),
        "R2_test":    r2_score(y_test,  y_pred_test),
        "MAE_test":   mean_absolute_error(y_test, y_pred_test),
    }

    print(f"\n{'='*40}")
    print(f"  {nom}")
    print(f"{'='*40}")
    print(f"  RMSE train : {resultats['RMSE_train']:.4f} t/ha")
    print(f"  RMSE test  : {resultats['RMSE_test']:.4f} t/ha")
    print(f"  R²   train : {resultats['R2_train']:.4f}")
    print(f"  R²   test  : {resultats['R2_test']:.4f}")
    print(f"  MAE  test  : {resultats['MAE_test']:.4f} t/ha")

    return resultats, modele.predict(X_test)

tous_resultats = []

In [5]:
# Cellule 5 — Régression linéaire
modele_lr = LinearRegression()
resultats_lr, y_pred_lr = evaluer_modele(
    "Régression Linéaire",
    modele_lr, X_train, X_test, y_train, y_test
)
tous_resultats.append(resultats_lr)


  Régression Linéaire
  RMSE train : 0.3039 t/ha
  RMSE test  : 0.3359 t/ha
  R²   train : 0.4514
  R²   test  : 0.3072
  MAE  test  : 0.2272 t/ha


In [6]:
# Cellule 6 — Random Forest
modele_rf = RandomForestRegressor(
    n_estimators=200, max_depth=10,
    min_samples_split=5, min_samples_leaf=2,
    random_state=42, n_jobs=-1
)
resultats_rf, y_pred_rf = evaluer_modele(
    "Random Forest",
    modele_rf, X_train, X_test, y_train, y_test
)
tous_resultats.append(resultats_rf)


  Random Forest
  RMSE train : 0.2156 t/ha
  RMSE test  : 0.3264 t/ha
  R²   train : 0.7237
  R²   test  : 0.3460
  MAE  test  : 0.2190 t/ha


In [7]:
# Cellule 7 — XGBoost régularisé
modele_xgb = xgb.XGBRegressor(
    n_estimators=200, learning_rate=0.03,
    max_depth=3, subsample=0.7,
    colsample_bytree=0.6, min_child_weight=5,
    reg_alpha=0.5, reg_lambda=2.0, gamma=0.2,
    random_state=42, n_jobs=-1, verbosity=0
)
resultats_xgb, y_pred_xgb = evaluer_modele(
    "XGBoost Régularisé",
    modele_xgb, X_train, X_test, y_train, y_test
)
tous_resultats.append(resultats_xgb)


  XGBoost Régularisé
  RMSE train : 0.2741 t/ha
  RMSE test  : 0.3285 t/ha
  R²   train : 0.5537
  R²   test  : 0.3375
  MAE  test  : 0.2229 t/ha


In [8]:
# Cellule 8 — sélection des top features
importances = pd.Series(
    modele_rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

top_features = importances.head(30).index.tolist()

print("Top 30 features :")
print(importances.head(30).round(4))

X_train_top = X_train[top_features]
X_test_top  = X_test[top_features]

# Réentraîner les trois modèles
modele_lr2 = LinearRegression()
resultats_lr2, y_pred_lr2 = evaluer_modele(
    "Régression Linéaire (top 30)",
    modele_lr2, X_train_top, X_test_top, y_train, y_test
)

modele_rf2 = RandomForestRegressor(
    n_estimators=200, max_depth=10,
    min_samples_split=5, min_samples_leaf=2,
    random_state=42, n_jobs=-1
)
resultats_rf2, y_pred_rf2 = evaluer_modele(
    "Random Forest (top 30)",
    modele_rf2, X_train_top, X_test_top, y_train, y_test
)

modele_xgb2 = xgb.XGBRegressor(
    n_estimators=200, learning_rate=0.03,
    max_depth=3, subsample=0.7,
    colsample_bytree=0.6, min_child_weight=5,
    reg_alpha=0.5, reg_lambda=2.0, gamma=0.2,
    random_state=42, n_jobs=-1, verbosity=0
)
resultats_xgb2, y_pred_xgb2 = evaluer_modele(
    "XGBoost Régularisé (top 30)",
    modele_xgb2, X_train_top, X_test_top, y_train, y_test
)

tous_resultats.extend([resultats_lr2, resultats_rf2, resultats_xgb2])

Top 30 features :
amplitude_annuelle_c         0.1017
humidite_11_pct              0.0790
humidite_moy_annuelle_pct    0.0692
departement_encode           0.0505
humidite_10_pct              0.0478
rayonnement_11               0.0477
annee                        0.0234
annee_normalisee             0.0223
temp_min_07_c                0.0155
rayonnement_moyen_annuel     0.0155
humidite_12_pct              0.0135
precip_12_mm                 0.0131
precip_02_mm                 0.0115
type_sol_encode              0.0114
temp_max_02_c                0.0109
rayonnement_04               0.0109
rayonnement_09               0.0105
amplitude_05_c               0.0104
rayonnement_06               0.0097
amplitude_11_c               0.0091
precip_09_mm                 0.0089
rayonnement_03               0.0086
temp_max_05_c                0.0085
humidite_02_pct              0.0078
temp_max_annuelle_c          0.0078
humidite_05_pct              0.0076
rayonnement_10               0.0075
humidite_0

In [9]:
# Cellule 9 — encoder la commune directement
from sklearn.preprocessing import LabelEncoder

le_commune = LabelEncoder()
dataset["commune_encode"] = le_commune.fit_transform(dataset["commune"])

# Recréer X avec la commune encodée
cols_exclure = ["commune", "departement", "type_sol", "rendement_kg_ha", "rendement_t_ha"]
X2 = dataset.drop(columns=cols_exclure)
y2 = dataset["rendement_t_ha"]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42
)

# Random Forest avec commune encodée
modele_rf3 = RandomForestRegressor(
    n_estimators=300, max_depth=10,
    min_samples_split=5, min_samples_leaf=2,
    random_state=42, n_jobs=-1
)
resultats_rf3, y_pred_rf3 = evaluer_modele(
    "Random Forest + commune",
    modele_rf3, X2_train, X2_test, y2_train, y2_test
)
tous_resultats.append(resultats_rf3)

# XGBoost avec commune encodée
modele_xgb3 = xgb.XGBRegressor(
    n_estimators=200, learning_rate=0.03,
    max_depth=3, subsample=0.7,
    colsample_bytree=0.6, min_child_weight=5,
    reg_alpha=0.5, reg_lambda=2.0, gamma=0.2,
    random_state=42, n_jobs=-1, verbosity=0
)
resultats_xgb3, y_pred_xgb3 = evaluer_modele(
    "XGBoost + commune",
    modele_xgb3, X2_train, X2_test, y2_train, y2_test
)
tous_resultats.append(resultats_xgb3)


  Random Forest + commune
  RMSE train : 0.1878 t/ha
  RMSE test  : 0.3190 t/ha
  R²   train : 0.7904
  R²   test  : 0.3755
  MAE  test  : 0.2119 t/ha

  XGBoost + commune
  RMSE train : 0.2666 t/ha
  RMSE test  : 0.3235 t/ha
  R²   train : 0.5779
  R²   test  : 0.3576
  MAE  test  : 0.2215 t/ha


In [10]:
# Cellule 10 — prédire l'anomalie de rendement
# Calculer la moyenne historique par commune
moyenne_commune = dataset.groupby("commune")["rendement_t_ha"].mean()
dataset["rendement_moyen_commune"] = dataset["commune"].map(moyenne_commune)
dataset["anomalie_rendement"] = dataset["rendement_t_ha"] - dataset["rendement_moyen_commune"]

print("Distribution des anomalies :")
print(dataset["anomalie_rendement"].describe().round(3))

# Recréer X et y avec l'anomalie comme cible
cols_exclure = ["commune", "departement", "type_sol",
                "rendement_kg_ha", "rendement_t_ha",
                "rendement_moyen_commune", "anomalie_rendement"]
X3 = dataset.drop(columns=cols_exclure)
y3 = dataset["anomalie_rendement"]

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3, test_size=0.2, random_state=42
)

modele_rf4 = RandomForestRegressor(
    n_estimators=300, max_depth=8,
    min_samples_split=5, min_samples_leaf=2,
    random_state=42, n_jobs=-1
)
resultats_rf4, y_pred_rf4 = evaluer_modele(
    "Random Forest — anomalie",
    modele_rf4, X3_train, X3_test, y3_train, y3_test
)

modele_xgb4 = xgb.XGBRegressor(
    n_estimators=200, learning_rate=0.03,
    max_depth=3, subsample=0.7,
    colsample_bytree=0.6, min_child_weight=5,
    reg_alpha=0.5, reg_lambda=2.0, gamma=0.2,
    random_state=42, n_jobs=-1, verbosity=0
)
resultats_xgb4, y_pred_xgb4 = evaluer_modele(
    "XGBoost — anomalie",
    modele_xgb4, X3_train, X3_test, y3_train, y3_test
)

tous_resultats.extend([resultats_rf4, resultats_xgb4])

Distribution des anomalies :
count    1514.000
mean       -0.000
std         0.311
min        -1.779
25%        -0.156
50%        -0.010
75%         0.141
max         2.696
Name: anomalie_rendement, dtype: float64

  Random Forest — anomalie
  RMSE train : 0.1904 t/ha
  RMSE test  : 0.2898 t/ha
  R²   train : 0.6160
  R²   test  : 0.1919
  MAE  test  : 0.1939 t/ha

  XGBoost — anomalie
  RMSE train : 0.2390 t/ha
  RMSE test  : 0.2944 t/ha
  R²   train : 0.3955
  R²   test  : 0.1660
  MAE  test  : 0.1987 t/ha


In [11]:
# Cellule 11 — tableau comparatif final
df_resultats = pd.DataFrame(tous_resultats).round(4)
print(df_resultats[["Modele","RMSE_test","R2_test","MAE_test"]].to_string(index=False))

                      Modele  RMSE_test  R2_test  MAE_test
         Régression Linéaire     0.3359   0.3072    0.2272
               Random Forest     0.3264   0.3460    0.2190
          XGBoost Régularisé     0.3285   0.3375    0.2229
Régression Linéaire (top 30)     0.3568   0.2185    0.2458
      Random Forest (top 30)     0.3258   0.3484    0.2172
 XGBoost Régularisé (top 30)     0.3256   0.3492    0.2214
     Random Forest + commune     0.3190   0.3755    0.2119
           XGBoost + commune     0.3235   0.3576    0.2215
    Random Forest — anomalie     0.2898   0.1919    0.1939
          XGBoost — anomalie     0.2944   0.1660    0.1987


In [12]:
# Cellule 12 — sauvegarde du meilleur modèle communal
import os
os.makedirs("../models/trained_models", exist_ok=True)

# Meilleur modèle : Random Forest + commune
joblib.dump(modele_rf3, "../models/trained_models/random_forest_communal.joblib")

# Sauvegarder les features utilisées
top_features_communal = X2_train.columns.tolist()
joblib.dump(top_features_communal, "../models/trained_models/top_features_communal.joblib")

# Sauvegarder les moyennes par commune pour l'interface web
moyenne_commune_df = dataset.groupby("commune")["rendement_t_ha"].mean().reset_index()
moyenne_commune_df.columns = ["commune", "rendement_moyen_t_ha"]
moyenne_commune_df.to_csv("../data/processed/moyenne_rendement_commune.csv", index=False)

# Tableau comparatif
df_resultats.to_csv("../models/model_evaluation/comparaison_modeles_communal.csv", index=False)

print("Modèle communal sauvegardé.")
print(f"Meilleur modèle : Random Forest + commune")
print(f"R² test  : 0.3755")
print(f"RMSE test: 0.3190 t/ha")
print("\n=== COMPARAISON FINALE ===")
print(f"Dataset départemental — Random Forest top 30 : R²=0.591, RMSE=0.210 t/ha")
print(f"Dataset communal      — Random Forest+commune : R²=0.376, RMSE=0.319 t/ha")
print("\nConclusion : le modèle départemental reste le meilleur pour le déploiement.")

Modèle communal sauvegardé.
Meilleur modèle : Random Forest + commune
R² test  : 0.3755
RMSE test: 0.3190 t/ha

=== COMPARAISON FINALE ===
Dataset départemental — Random Forest top 30 : R²=0.591, RMSE=0.210 t/ha
Dataset communal      — Random Forest+commune : R²=0.376, RMSE=0.319 t/ha

Conclusion : le modèle départemental reste le meilleur pour le déploiement.
